# sample5 - CPU / GPU 切り替え

PyTorch では `.to(device)` の一行で CPU / GPU を切り替えられます。  
GPU がなくても動作するコードの書き方を習得します。

| ステップ | 内容 |
|----------|------|
| 1 | デバイスの確認 |
| 2 | テンソルを GPU に移動 |
| 3 | モデルを GPU に移動 |
| 4 | GPU対応の学習ループ |
| 5 | CPU / GPU の速度比較 |

## 1. デバイスの確認

In [1]:
import torch
import torch.nn as nn
import time

# GPU があれば cuda、なければ cpu を自動選択
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用デバイス:", device)

if torch.cuda.is_available():
    print("GPU名:", torch.cuda.get_device_name(0))
    print("GPUメモリ:", torch.cuda.get_device_properties(0).total_memory // 1024**2, "MB")
else:
    print("GPU は使用できません。CPU で実行します。")

使用デバイス: cuda
GPU名: NVIDIA GeForce RTX 5060 Ti
GPUメモリ: 16310 MB


## 2. テンソルを GPU に移動

In [2]:
# CPU上のテンソル
t_cpu = torch.randn(3, 3)
print("CPU テンソル device:", t_cpu.device)

# デバイスに移動（GPU があれば GPU、なければ CPU のまま）
t_device = t_cpu.to(device)
print("移動後 device:", t_device.device)

# 演算はデバイスが一致している必要がある
t2 = torch.randn(3, 3).to(device)
result = t_device + t2
print("演算結果 device:", result.device)

CPU テンソル device: cpu
移動後 device: cuda:0
演算結果 device: cuda:0


## 3. モデルを GPU に移動

In [3]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

# モデルをデバイスに移動
model = SimpleNet().to(device)

# パラメータがどのデバイスにあるか確認
for name, param in model.named_parameters():
    print(f"{name}: {param.device}")

net.0.weight: cuda:0
net.0.bias: cuda:0
net.2.weight: cuda:0
net.2.bias: cuda:0


## 4. GPU 対応の学習ループ

ポイントは **データとモデルを同じデバイスに置く** ことです。

In [4]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# データをデバイスに送る
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
y_test  = torch.tensor(y_test,  dtype=torch.long).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    pred = torch.argmax(model(X_test), dim=1)
    acc  = (pred == y_test).float().mean()
    print(f"デバイス: {device} | 正解率: {acc.item():.4f}")

デバイス: cuda | 正解率: 1.0000


## 5. CPU / GPU の速度比較

大きな行列積で CPU と GPU の速度を比較します。

In [5]:
size = 4096
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

# CPU
start = time.time()
_ = torch.mm(a_cpu, b_cpu)
cpu_time = time.time() - start
print(f"CPU 時間: {cpu_time:.4f} 秒")

if torch.cuda.is_available():
    a_gpu = a_cpu.to('cuda')
    b_gpu = b_cpu.to('cuda')
    torch.cuda.synchronize()  # GPU の処理完了を待つ

    start = time.time()
    _ = torch.mm(a_gpu, b_gpu)
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"GPU 時間: {gpu_time:.4f} 秒")
    print(f"高速化: {cpu_time / gpu_time:.1f} 倍")
else:
    print("GPU なし: CPU のみで実行")

CPU 時間: 0.1162 秒
GPU 時間: 0.0140 秒
高速化: 8.3 倍
